# (Opsiyonal) Olist reviews' - Translations...

* 🇧🇷 Brezilya Portekizcesi bilmiyorsanız, hadi `order_reviews` metinlerini 🇬🇧 İngilizce’ye çevirelim!

* Bunun için `pip install googletrans==4.0.2` yüklemeniz gerekecek.

☢️ Bu API ile herhangi bir sorun yaşarsanız, bunu düzeltmek için zaman harcamayın, bu package oldukça dengesiz… Aklınızda bulunsun:
- bu optional bir challenge
- Brezilya Portekizcesi ile yazılmış review’ların anlamını görmek için bazı yorumları Google Translate’e kopyalayıp yapıştırarak yine de eski yöntemle yapabilirsiniz.

## Review Translator

👉 `reviews` dataset’ini load edin ve 1-yıldız rating’e sahip review’lardan bir örnek (sample) seçin.

In [1]:
# Verileri yükle
from olist.data import Olist
data = Olist().get_data()

👀 20 adet düşük puan alan yorumdan oluşan bir örneklem seçin (rastgele) ve bunu bir listeye dönüştürün.

In [2]:
import pandas as pd

# 1. Verileri yükleyelim (Data class'ını kullandığını varsayıyoruz)
reviews = data['order_reviews'].copy()

# 2. Sadece 1 yıldızlı (review_score == 1) yorumları filtreleyelim
# Ayrıca yorum metni boş (NaN) olanları çıkaralım ki çeviri hata vermesin
bad_reviews = reviews[(reviews['review_score'] == 1) & (reviews['review_comment_message'].notna())]

# 3. Rastgele 20 adet örnek seçelim
sample_reviews = bad_reviews.sample(20, random_state=42)

# 4. Bu yorumları bir listeye dönüştürelim
reviews_list = sample_reviews['review_comment_message'].tolist()

# Kontrol için ilk birkaç tanesine bakalım
print(f"Toplam 1 yıldızlı ve metni olan yorum sayısı: {len(bad_reviews)}")
print(f"Seçilen örnek sayısı: {len(reviews_list)}")
print("-" * 30)
for i, msg in enumerate(reviews_list[:3]):
    print(f"{i+1}. Portekizce Yorum: {msg}")

Toplam 1 yıldızlı ve metni olan yorum sayısı: 8745
Seçilen örnek sayısı: 20
------------------------------
1. Portekizce Yorum: Foi enviado produto Diferente daquele do anúncio. Solicitei devolver o produto e receber o valor pago, porém ainda não tive resposta. Espero resolver este desagradável problema de forma consensual.
2. Portekizce Yorum: Não comprem desta loja! Tentei fazer o cancelamento da compra por 5 vezes e só fui atende depois de ter recusado a entrega do Correio.
3. Portekizce Yorum: Excelente


In [4]:
!pip install googletrans==4.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [googletrans]

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [9]:
!pip install deep-translator


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


🗣 Bu göreve başlamadan önce önceden yüklediğiniz `google_translator` paketini kullanarak bu yorumları çevirin:

In [10]:
from deep_translator import GoogleTranslator

# 1. Çeviriciyi hazırlayalım (Portekizce'den İngilizce'ye)
translator = GoogleTranslator(source='pt', target='en')

translated_reviews = []

print("20 yorumun çevirisi başlıyor...")

# 2. Senin o meşhur 20'li listenin üzerinden geçelim
for i, text in enumerate(reviews_list):
    try:
        # Çeviri işlemini yap
        translation = translator.translate(text)
        translated_reviews.append(translation)
        print(f"{i+1}. Çeviri Başarılı")
    except Exception as e:
        print(f"{i+1}. yorumda hata: {e}")
        translated_reviews.append("Translation Error")

# 3. Sonuçları bir DataFrame'e koyup düzgünce görelim
import pandas as pd
df_final = pd.DataFrame({
    'Portekizce': reviews_list,
    'Ingilizce': translated_reviews
})

# Hepsini görmek için
pd.set_option('display.max_colwidth', None)
print(df_final)

20 yorumun çevirisi başlıyor...


/tmp/ipykernel_5613/3738083303.py:14: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  translation = translator.translate(text)


1. Çeviri Başarılı
2. Çeviri Başarılı
3. Çeviri Başarılı
4. Çeviri Başarılı
5. Çeviri Başarılı
6. Çeviri Başarılı
7. Çeviri Başarılı
8. Çeviri Başarılı
9. Çeviri Başarılı
10. Çeviri Başarılı
11. Çeviri Başarılı
12. Çeviri Başarılı
13. Çeviri Başarılı
14. Çeviri Başarılı
15. Çeviri Başarılı
16. Çeviri Başarılı
17. Çeviri Başarılı
18. Çeviri Başarılı
19. Çeviri Başarılı
20. Çeviri Başarılı
                                                                                                                                                                                                Portekizce  \
0    Foi enviado produto Diferente daquele do anúncio. Solicitei devolver o produto e receber o valor pago, porém ainda não tive resposta. Espero resolver este desagradável problema de forma consensual.   
1                                                                   Não comprem desta loja! Tentei fazer o cancelamento da compra por 5 vezes e só fui atende depois de ter recusado a entrega do Cor

**Insights** 💡
- Bazı kötü review’lar delivery ile ilgili (`wait_time`, kaçırılan teslim tarihi, vb.)
- Ancak bazı kötü review’lar seller veya ürünle ilgili...

Peki iki olası nedeni nasıl ayırt edebiliriz?

Bu Olist’in mutlaka bilmesi gereken bir şey:
- bazı ürünleri katalogdan mı çıkarmalı?
- yoksa bazı seller’ları marketplace’ten mi kaldırmalı?
